# 1. Introduction
## Feature Engineering

### Objectif

Cette étape consiste à créer des variables explicatives pertinentes à partir des données historiques des commandes.

Le but est d’améliorer les performances des modèles prédictifs en capturant :
- les comportements temporels ;
- les habitudes de commande ;
- les tendances des stations-service.

# 2. Importation des Bibliothèques

In [28]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder

# Configuration affichage
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1200)

# 3. Chargement des données propres

In [29]:
df = pd.read_excel("../data/propre/commandes_propres.xlsx")

print(df.shape)

display(df.head())

(34150, 13)


,Date comptabilisation,N° article,N° origine,N° contact origine,Quantité facturée,Code Produit,Type document,NOM CLIENT,Nom sous compte,TYPE,STATUT,Code Station,GROUPE TYPE
0,2020-01-13,A0001,C1234,CC2616,5000,PROD-BLANC,Expédition vente,GIBLAND COMPAGNY,STATION ATTAKE,CODO,Fonctionnelle,ST0049,STATION
1,2020-01-13,A0002,C1234,CC2616,4000,PROD-BLANC,Expédition vente,GIBLAND COMPAGNY,STATION ATTAKE,CODO,Fonctionnelle,ST0049,STATION
2,2020-01-13,A0001,C1249,CC2650,1000,PROD-BLANC,Expédition vente,TONI ET FILS,STATION NDALI,DODO,Fonctionnelle,ST0092,STATION
3,2020-01-13,A0002,C1249,CC2650,16000,PROD-BLANC,Expédition vente,TONI ET FILS,STATION NDALI,DODO,Fonctionnelle,ST0092,STATION
4,2020-01-13,A0001,C1233,CC2614,5000,PROD-BLANC,Expédition vente,AYODELE BUSINESS,STATION OUIDAH,CODO,Fonctionnelle,ST0016,STATION


# 4. Vérification du format des dates
Les modèles temporels ont besoin : de dates exploitables et d’ordre chronologique.

In [30]:
# CONVERSION DES DATES

df["Date comptabilisation"] = pd.to_datetime(
    df["Date comptabilisation"]
)

print(df["Date comptabilisation"].dtype)

datetime64[ns]


In [31]:
# TRI CHRONOLOGIQUE
# Les variables historiques dépendent : de l’ordre temporel et des commandes précédentes.

df = df.sort_values(
    ["Code Station", "Date comptabilisation"]
)

display(df.head())

,Date comptabilisation,N° article,N° origine,N° contact origine,Quantité facturée,Code Produit,Type document,NOM CLIENT,Nom sous compte,TYPE,STATUT,Code Station,GROUPE TYPE
13322,2022-09-27,A0001,C0083,CC6471,20000,PROD-BLANC,Expédition vente,2M ASSOCIES,STATION BOULEVARD MARINA,CODO,Fonctionnelle,ST0001,STATION
13323,2022-09-27,A0002,C0083,CC6471,15000,PROD-BLANC,Expédition vente,2M ASSOCIES,STATION BOULEVARD MARINA,CODO,Fonctionnelle,ST0001,STATION
13637,2022-10-17,A0001,C0083,CC6471,8000,PROD-BLANC,Expédition vente,2M ASSOCIES,STATION BOULEVARD MARINA,CODO,Fonctionnelle,ST0001,STATION
13648,2022-10-17,A0002,C0083,CC6471,10000,PROD-BLANC,Expédition vente,2M ASSOCIES,STATION BOULEVARD MARINA,CODO,Fonctionnelle,ST0001,STATION
13727,2022-10-20,A0002,C0083,CC6471,8000,PROD-BLANC,Expédition vente,2M ASSOCIES,STATION BOULEVARD MARINA,CODO,Fonctionnelle,ST0001,STATION


In [32]:
# AGREGER LES COMMANDES JOURNALIERES

df_daily = (
    df.groupby(
        [
            "Date comptabilisation",
            "Code Station",
            "N° article"
        ]
    )
    .agg({
        "Quantité facturée": "sum"
    })
    .reset_index()
    .copy()
)

In [33]:
all_dates = pd.date_range(
    start=df_daily["Date comptabilisation"].min(),
    end=df_daily["Date comptabilisation"].max(),
    freq="D"
)

In [34]:
df_daily = df_daily.sort_values(
    [
        "Code Station",
        "N° article",
        "Date comptabilisation"
    ]
).copy()

In [35]:
display(df_daily.head(20))

,Date comptabilisation,Code Station,N° article,Quantité facturée
10703,2022-09-27,ST0001,A0001,20000
10959,2022-10-17,ST0001,A0001,8000
11042,2022-10-24,ST0001,A0001,6000
11079,2022-10-25,ST0001,A0001,10000
11143,2022-11-03,ST0001,A0001,15000
11228,2022-11-08,ST0001,A0001,12000
11283,2022-11-11,ST0001,A0001,22000
12009,2023-01-05,ST0001,A0001,10000
12073,2023-01-12,ST0001,A0001,12000
12110,2023-01-16,ST0001,A0001,10000


# 5. Création des variables temporelles


In [39]:
#Certains jours génèrent plus ou moins de commandes
df_daily["Jour"] = df_daily["Date comptabilisation"].dt.day

# Détection de saisonnalité et variations mensuelles
df_daily["Mois"] = df_daily["Date comptabilisation"].dt.month

# Evolution globale et croissance de l'activité
df_daily["Annee"] = df_daily["Date comptabilisation"].dt.year

# Les comportement diffèrent lundi, fin de semaine et week-end
df_daily["JourSemaine"] = (
    df_daily["Date comptabilisation"]
    .dt.dayofweek
)

df_daily["Weekend"] = (
    df_daily["JourSemaine"]
    .isin([5, 6])
    .astype(int)
)
#Fin du mois
df_daily["FinMois"] = (
    df_daily["Date comptabilisation"]
    .dt.is_month_end
    .astype(int)
)

#Debut Mois
df_daily["DebutMois"] = (
    df_daily["Date comptabilisation"]
    .dt.is_month_start
    .astype(int)
)

#Trimestre
df_daily["Trimestre"] = (
    df_daily["Date comptabilisation"]
    .dt.quarter
)

# 6. Création des variables historiques

## 6.1 Quantité précédente (Lag_1, Lag_7 et Lag_30)

La dernière commande influence fortement la prochaine commande.

In [43]:
 # LAGS TEMPORELS
group_cols = ["Code Station", "N° article"]
df_daily["Lag_1"] = (
    df_daily.groupby(group_cols)
    ["Quantité facturée"]
    .shift(1)
)
df_daily["Lag_7"] = (
    df_daily.groupby(group_cols)
    ["Quantité facturée"]
    .shift(7)
)
df_daily["Lag_30"] = (
    df_daily.groupby(group_cols)
    ["Quantité facturée"]
    .shift(30)
)

In [44]:
# RÉSUMÉ LAGS — 2 lignes par (station, article)
# Ligne 1 : moyenne des valeurs calculées pour chaque lag
# Ligne 2 : bilan des NaN (valeurs manquantes, normales en début de série)

def resume_lags(df_daily, group_cols):
    resultats = []

    for (station, article), grp in df_daily.groupby(group_cols):
        n = len(grp)

        # ── Ligne 1 : statistiques des lags (valeurs moyennes) ────────────
        row_stats = {
            'Code Station' : station,
            'N° article'   : article,
            'Type'         : 'Valeurs (moyenne)',
            'Nb jours'     : n,
            'Qté moy.'     : round(grp['Quantité facturée'].mean(), 1),
            'Lag_1'        : round(grp['Lag_1'].mean(),  1),
            'Lag_7'        : round(grp['Lag_7'].mean(),  1),
            'Lag_30'       : round(grp['Lag_30'].mean(), 1),
        }

        # ── Ligne 2 : bilan NaN (attendus au début de chaque série) ───────
        # Lag_1 → 1 NaN, Lag_7 → 7 NaN, Lag_30 → 30 NaN par groupe
        row_nan = {
            'Code Station' : station,
            'N° article'   : article,
            'Type'         : 'NaN (début série)',
            'Nb jours'     : n,
            'Qté moy.'     : '-',
            'Lag_1'        : f"{grp['Lag_1'].isna().sum()} / {n}",
            'Lag_7'        : f"{grp['Lag_7'].isna().sum()} / {n}",
            'Lag_30'       : f"{grp['Lag_30'].isna().sum()} / {n}",
        }

        resultats.extend([row_stats, row_nan])

    return pd.DataFrame(resultats)


resume = resume_lags(df_daily, group_cols)
resume

,Code Station,N° article,Type,Nb jours,Qté moy.,Lag_1,Lag_7,Lag_30
0,ST0001,A0001,Valeurs (moyenne),237,13240.5,13211.9,13426.1,13724.6
1,ST0001,A0001,NaN (début série),237,-,1 / 237,7 / 237,30 / 237
2,ST0001,A0002,Valeurs (moyenne),163,11521.5,11506.2,11679.5,12112.8
3,ST0001,A0002,NaN (début série),163,-,1 / 163,7 / 163,30 / 163
4,ST0002,A0001,Valeurs (moyenne),660,23971.2,23984.8,24030.6,24481.0
...,...,...,...,...,...,...,...,...
369,ST0094,A0002,NaN (début série),61,-,1 / 61,7 / 61,30 / 61
370,ST0095,A0001,Valeurs (moyenne),40,11400.0,11333.3,11515.2,11100.0
371,ST0095,A0001,NaN (début série),40,-,1 / 40,7 / 40,30 / 40
372,ST0095,A0002,Valeurs (moyenne),23,5565.2,5590.9,5812.5,NaN


**Lag_30 pose trois problèmes**  
**Problème 1** — Certaines stations ont trop peu d'historique
Lag_30 cherche la valeur d'il y a 30 jours. Si une station comme ST0095/A0002 n'a que 23 jours de données, cette valeur n'existe tout simplement pas. Lag_30 sera NaN sur toute la série de cette station — la feature est inutilisable pour elle.  
**Problème 2** — Certaines stations ne commandent pas tous les jours
Si une station commande de façon irrégulière (lundi, jeudi, lundi suivant…), "il y a 30 jours" ne tombe pas forcément sur un jour de commande. Tu récupères soit un NaN, soit une valeur d'un jour qui n'a rien à voir avec la situation actuelle. Le lag perd son sens.

In [48]:
# LAGS TEMPORELS — version finale

# Lag_1 : hier → effet immédiat
df_daily["Lag_1"] = (
    df_daily.groupby(group_cols)["Quantité facturée"].shift(1)
)

# Lag_7 : même jour semaine passée → pattern hebdomadaire
df_daily["Lag_7"] = (
    df_daily.groupby(group_cols)["Quantité facturée"].shift(7)
)

# Supprimer Lag_30 si tu l'avais déjà créé
if "Lag_30" in df_daily.columns:
    df_daily.drop(columns=["Lag_30"], inplace=True)

print("Lags retenus : Lag_1 et Lag_7 uniquement")
print(f"NaN Lag_1 : {df_daily['Lag_1'].isna().sum()} (1 par groupe — normal)")
print(f"NaN Lag_7 : {df_daily['Lag_7'].isna().sum()} (7 par groupe — normal)")

Lags retenus : Lag_1 et Lag_7 uniquement
NaN Lag_1 : 187 (1 par groupe — normal)
NaN Lag_7 : 1290 (7 par groupe — normal)


## 6.3 Moyenne mobile sur 7 et 30 jours

Capture tendance récente et stabilité des commandes.

In [49]:
# ROLLING MEAN — version corrigée 
group_cols = ["Code Station", "N° article"]

# Rolling 7 jours : moyenne des 7 jours PRÉCÉDENTS (jour actuel exclu)
df_daily["Rolling_7j"] = (
    df_daily.groupby(group_cols)["Quantité facturée"]
    .transform(lambda x: x.shift(1).rolling(7, min_periods=3).mean())
)

# Rolling 30 jours : moyenne des 30 jours PRÉCÉDENTS (jour actuel exclu)
df_daily["Rolling_30j"] = (
    df_daily.groupby(group_cols)["Quantité facturée"]
    .transform(lambda x: x.shift(1).rolling(30, min_periods=7).mean())
)

## 6.4 Écart-type mobile

Mesure variabilité et instabilité des commandes.

In [50]:
# ROLLING STD

# Écart-type sur 7 jours précédents (volatilité court terme)
df_daily["Rolling_Std_7j"] = (
    df_daily.groupby(group_cols)["Quantité facturée"]
    .transform(lambda x: x.shift(1).rolling(7, min_periods=3).std())
)

# Écart-type sur 30 jours précédents (volatilité long terme)
df_daily["Rolling_Std_30j"] = (
    df_daily.groupby(group_cols)["Quantité facturée"]
    .transform(lambda x: x.shift(1).rolling(30, min_periods=7).std())
)


## 7. Encodage des variables catégorielles
Les modèles ML ne comprennent pas le texte.

In [51]:
# ENCODAGE STATIONS

le_station = LabelEncoder()

df_daily["StationEncoded"] = le_station.fit_transform(
    df_daily["Code Station"]
)

# ENCODAGE PRODUIT transformer catégories en valeures numérique

le_produit = LabelEncoder()

df_daily["ProduitEncoded"] = le_produit.fit_transform(
    df_daily["N° article"]
)

## 9. Vérification des valeurs manquantes

Les lags peuvent créent des NaN

In [52]:
# VALEURS MANQUANTES

print(df_daily.isnull().sum())

print(df_daily.shape)

Date comptabilisation       0
Code Station                0
N° article                  0
Quantité facturée           0
Jour                        0
Mois                        0
Annee                       0
JourSemaine                 0
Weekend                     0
FinMois                     0
DebutMois                   0
Trimestre                   0
Lag_1                     187
Lag_7                    1290
Rolling_7j                558
Rolling_30j              1290
Rolling_Std_7j            558
Rolling_Std_30j          1290
StationEncoded              0
ProduitEncoded              0
dtype: int64
(27071, 20)


In [53]:
FEATURES = [
    # Identifiants encodés
    "StationEncoded",
    "ProduitEncoded",

    # Temporelles
    "Mois",
    "Weekend",                     
    "JourSemaine",
    "FinMois",

    # Lags
    "Lag_1",
    "Lag_7",

    # Rolling mean
    "Rolling_7j",
    "Rolling_30j",

    # Rolling std (volatilité)
    "Rolling_Std_7j",
    "Rolling_Std_30j",
]

TARGET = "Quantité facturée"

In [55]:
# APERÇU FINAL DU DATASET AVANT NETTOYAGE
print("ÉTAT DU DATASET AVANT NETTOYAGE")
print(f"Dimensions        : {df_daily.shape[0]:,} lignes x {df_daily.shape[1]} colonnes")
print(f"Période           : {df_daily['Date comptabilisation'].min().date()} → {df_daily['Date comptabilisation'].max().date()}")
print(f"Stations          : {df_daily['Code Station'].nunique()}")
print(f"Articles          : {df_daily['N° article'].nunique()}")
print()
print("NaN par colonne :")
print(df_daily[FEATURES + [TARGET]].isnull().sum().to_string())

ÉTAT DU DATASET AVANT NETTOYAGE
Dimensions        : 27,071 lignes x 20 colonnes
Période           : 2020-01-13 → 2025-12-30
Stations          : 94
Articles          : 2

NaN par colonne :
StationEncoded          0
ProduitEncoded          0
Mois                    0
Weekend                 0
JourSemaine             0
FinMois                 0
Lag_1                 187
Lag_7                1290
Rolling_7j            558
Rolling_30j          1290
Rolling_Std_7j        558
Rolling_Std_30j      1290
Quantité facturée       0


In [56]:
# SUPPRESSION DES NaN

# On supprime toutes les lignes où au moins une feature est NaN
# Lag_7 étant le lag le plus long retenu, toute ligne valide pour Lag_7
# l'est aussi pour toutes les autres features
df_ml = df_daily.dropna(subset=FEATURES).copy()
df_ml = df_ml.reset_index(drop=True)

print(f"Lignes supprimées : {len(df_daily) - len(df_ml):,}")
print(f"Lignes conservées : {len(df_ml):,}")

Lignes supprimées : 1,290
Lignes conservées : 25,781


## 10. Sélection des variables ML

C'est une étape est importante où je définis ce que le modèle verra et ce qu’il devra prédire.

In [57]:
# VÉRIFICATION FINALE
print("ÉTAT DU DATASET APRÈS NETTOYAGE")
print(f"Dimensions        : {df_ml.shape[0]:,} lignes x {df_ml.shape[1]} colonnes")
print(f"Période           : {df_ml['Date comptabilisation'].min().date()} → {df_ml['Date comptabilisation'].max().date()}")
print(f"Stations          : {df_ml['Code Station'].nunique()}")
print(f"Articles          : {df_ml['N° article'].nunique()}")
print()

nan_restants = df_ml[FEATURES + [TARGET]].isnull().sum()
if nan_restants.sum() == 0:
    print("NaN restants : aucun — dataset propre pour la ML")
else:
    print("NaN restants :")
    print(nan_restants[nan_restants > 0])

print()
print("Aperçu final :")
df_ml[['Date comptabilisation', 'Code Station', 'N° article',
       TARGET, 'Lag_1', 'Lag_7', 'Rolling_7j', 'Rolling_30j',
       'Rolling_Std_7j', 'StationEncoded', 'ProduitEncoded']].head(10)

ÉTAT DU DATASET APRÈS NETTOYAGE
Dimensions        : 25,781 lignes x 20 colonnes
Période           : 2020-01-24 → 2025-12-30
Stations          : 93
Articles          : 2

NaN restants : aucun — dataset propre pour la ML

Aperçu final :


,Date comptabilisation,Code Station,N° article,Quantité facturée,Lag_1,Lag_7,Rolling_7j,Rolling_30j,Rolling_Std_7j,StationEncoded,ProduitEncoded
0,2023-01-05,ST0001,A0001,10000,22000.0,20000.0,13285.714286,13285.714286,6019.808572,0,0
1,2023-01-12,ST0001,A0001,12000,10000.0,8000.0,11857.142857,12875.000000,5304.984179,0,0
2,2023-01-16,ST0001,A0001,10000,12000.0,6000.0,12428.571429,12777.777778,5028.490259,0,0
3,2023-01-24,ST0001,A0001,10000,10000.0,10000.0,13000.000000,12500.000000,4358.898944,0,0
4,2023-01-30,ST0001,A0001,14000,10000.0,15000.0,13000.000000,12272.727273,4358.898944,0,0
5,2023-02-06,ST0001,A0001,14000,14000.0,12000.0,12857.142857,12416.666667,4298.393941,0,0
6,2023-02-14,ST0001,A0001,10000,14000.0,22000.0,13142.857143,12538.461538,4298.393941,0,0
7,2023-02-23,ST0001,A0001,8000,10000.0,10000.0,11428.571429,12357.142857,1902.379462,0,0
8,2023-02-27,ST0001,A0001,2000,8000.0,12000.0,11142.857143,12066.666667,2267.786838,0,0
9,2023-03-01,ST0001,A0001,10000,2000.0,10000.0,9714.285714,11437.500000,4070.801957,0,0


In [64]:
# ── ÉTAPE 1 : agréger par semaine ────────────────────────────────────────────
# Chaque semaine devient une ligne, même si la station a commandé
# 1 fois ou 3 fois dans la semaine

df_daily['Date comptabilisation'] = pd.to_datetime(df_daily['Date comptabilisation'])

# Numéro de semaine ISO (lundi au dimanche)
df_daily['semaine'] = df_daily['Date comptabilisation'].dt.to_period('W').dt.start_time

df_weekly = (
    df_daily
    .groupby(['Code Station', 'N° article', 'semaine'])
    .agg(
        quantite_semaine=('Quantité facturée', 'sum'),   # total de la semaine
        nb_commandes    =('Quantité facturée', 'count'), # nb de jours commandés
    )
    .reset_index()
    .sort_values(['Code Station', 'N° article', 'semaine'])
    .reset_index(drop=True)
)

print(f"Lignes df_daily   : {len(df_daily):,}")
print(f"Lignes df_weekly  : {len(df_weekly):,}")
print(f"Semaines couvertes : {df_weekly['semaine'].nunique()}")
print()
display(df_weekly.head(20))

Lignes df_daily   : 27,071
Lignes df_weekly  : 20,207
Semaines couvertes : 312



,Code Station,N° article,semaine,quantite_semaine,nb_commandes
0,ST0001,A0001,2022-09-26,20000,1
1,ST0001,A0001,2022-10-17,8000,1
2,ST0001,A0001,2022-10-24,16000,2
3,ST0001,A0001,2022-10-31,15000,1
4,ST0001,A0001,2022-11-07,34000,2
5,ST0001,A0001,2023-01-02,10000,1
6,ST0001,A0001,2023-01-09,12000,1
7,ST0001,A0001,2023-01-16,10000,1
8,ST0001,A0001,2023-01-23,10000,1
9,ST0001,A0001,2023-01-30,14000,1


In [63]:
# ── ÉTAPE 2 : recalculer toutes les features sur df_weekly ───────────────────
group_cols = ['Code Station', 'N° article']

# Lags (maintenant en semaines — cohérent !)
df_weekly['Lag_1'] = df_weekly.groupby(group_cols)['quantite_semaine'].shift(1)
df_weekly['Lag_7'] = df_weekly.groupby(group_cols)['quantite_semaine'].shift(7)
# Lag_7 = même semaine 7 semaines avant (~2 mois) — plus Lag_52 si assez d'historique

# Rolling (maintenant en semaines — cohérent !)
g = df_weekly.groupby(group_cols)['quantite_semaine']

df_weekly['Rolling_4sem']  = g.transform(
    lambda x: x.shift(1).rolling(4, min_periods=2).mean()   # 1 mois
)
df_weekly['Rolling_12sem'] = g.transform(
    lambda x: x.shift(1).rolling(12, min_periods=4).mean()  # 3 mois
)
df_weekly['Rolling_Std_4sem'] = g.transform(
    lambda x: x.shift(1).rolling(4, min_periods=2).std()
)

# Features temporelles
df_weekly['mois']          = df_weekly['semaine'].dt.month
df_weekly['semaine_annee'] = df_weekly['semaine'].dt.isocalendar().week.astype(int)
df_weekly['est_fin_mois']  = (df_weekly['semaine'].dt.day >= 25).astype(int)

# Encodage
from sklearn.preprocessing import LabelEncoder
df_weekly['StationEncoded'] = LabelEncoder().fit_transform(df_weekly['Code Station'])
df_weekly['ProduitEncoded']  = LabelEncoder().fit_transform(df_weekly['N° article'])

print("Features recalculées sur base hebdomadaire")

display(df_weekly.head(20))

Features recalculées sur base hebdomadaire


,Code Station,N° article,semaine,quantite_semaine,nb_commandes,Lag_1,Lag_7,Rolling_4sem,Rolling_12sem,Rolling_Std_4sem,mois,semaine_annee,est_fin_mois,StationEncoded,ProduitEncoded
0,ST0001,A0001,2022-09-26,20000,1,NaN,NaN,NaN,NaN,NaN,9,39,1,0,0
1,ST0001,A0001,2022-10-17,8000,1,20000.0,NaN,NaN,NaN,NaN,10,42,0,0,0
2,ST0001,A0001,2022-10-24,16000,2,8000.0,NaN,14000.000000,NaN,8485.281374,10,43,0,0,0
3,ST0001,A0001,2022-10-31,15000,1,16000.0,NaN,14666.666667,NaN,6110.100927,10,44,1,0,0
4,ST0001,A0001,2022-11-07,34000,2,15000.0,NaN,14750.000000,14750.000000,4991.659711,11,45,0,0,0
5,ST0001,A0001,2023-01-02,10000,1,34000.0,NaN,18250.000000,18600.000000,11086.778913,1,1,0,0,0
6,ST0001,A0001,2023-01-09,12000,1,10000.0,NaN,18750.000000,17166.666667,10500.000000,1,2,0,0,0
7,ST0001,A0001,2023-01-16,10000,1,12000.0,20000.0,17750.000000,16428.571429,11026.483271,1,3,0,0,0
8,ST0001,A0001,2023-01-23,10000,1,10000.0,8000.0,16500.000000,15625.000000,11704.699911,1,4,0,0,0
9,ST0001,A0001,2023-01-30,14000,1,10000.0,16000.0,10500.000000,15000.000000,1000.000000,1,5,1,0,0


In [69]:
# ── FEATURE ENGINEERING COMPLET SUR df_weekly ────────────────────────────────

group_cols = ['Code Station', 'N° article']

# S'assurer que le dataset est bien trié avant les calculs
df_weekly = df_weekly.sort_values(group_cols + ['semaine']).reset_index(drop=True)

# ── LAGS ─────────────────────────────────────────────────────────────────────
df_weekly['Lag_1'] = df_weekly.groupby(group_cols)['quantite_semaine'].shift(1)
df_weekly['Lag_7'] = df_weekly.groupby(group_cols)['quantite_semaine'].shift(7)

# ── ROLLING MEAN ──────────────────────────────────────────────────────────────
g = df_weekly.groupby(group_cols)['quantite_semaine']

df_weekly['Rolling_4sem']  = g.transform(
    lambda x: x.shift(1).rolling(4,  min_periods=2).mean()
)
df_weekly['Rolling_12sem'] = g.transform(
    lambda x: x.shift(1).rolling(12, min_periods=4).mean()
)

# ── ROLLING STD ───────────────────────────────────────────────────────────────
df_weekly['Rolling_Std_4sem'] = g.transform(
    lambda x: x.shift(1).rolling(4, min_periods=2).std()
)

# ── FEATURES TEMPORELLES ──────────────────────────────────────────────────────
df_weekly['mois']          = df_weekly['semaine'].dt.month
df_weekly['semaine_annee'] = df_weekly['semaine'].dt.isocalendar().week.astype(int)
df_weekly['est_fin_mois']  = (df_weekly['semaine'].dt.day >= 25).astype(int)

# ── ENCODAGE ──────────────────────────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder

le_station = LabelEncoder()
le_produit  = LabelEncoder()

df_weekly['StationEncoded'] = le_station.fit_transform(df_weekly['Code Station'])
df_weekly['ProduitEncoded']  = le_produit.fit_transform(df_weekly['N° article'])

# ── VÉRIFICATION ──────────────────────────────────────────────────────────────
print(f"Colonnes disponibles : {df_weekly.shape[1]}")
print(df_weekly.columns.tolist())
print()
display(df_weekly.head())

Colonnes disponibles : 15
['Code Station', 'N° article', 'semaine', 'quantite_semaine', 'nb_commandes', 'Lag_1', 'Lag_7', 'Rolling_4sem', 'Rolling_12sem', 'Rolling_Std_4sem', 'mois', 'semaine_annee', 'est_fin_mois', 'StationEncoded', 'ProduitEncoded']



,Code Station,N° article,semaine,quantite_semaine,nb_commandes,Lag_1,Lag_7,Rolling_4sem,Rolling_12sem,Rolling_Std_4sem,mois,semaine_annee,est_fin_mois,StationEncoded,ProduitEncoded
0,ST0001,A0001,2022-09-26,20000,1,NaN,NaN,NaN,NaN,NaN,9,39,1,0,0
1,ST0001,A0001,2022-10-17,8000,1,20000.0,NaN,NaN,NaN,NaN,10,42,0,0,0
2,ST0001,A0001,2022-10-24,16000,2,8000.0,NaN,14000.000000,NaN,8485.281374,10,43,0,0,0
3,ST0001,A0001,2022-10-31,15000,1,16000.0,NaN,14666.666667,NaN,6110.100927,10,44,1,0,0
4,ST0001,A0001,2022-11-07,34000,2,15000.0,NaN,14750.000000,14750.0,4991.659711,11,45,0,0,0


In [70]:
# NETTOYAGE ET EXPORT
FEATURES = [
    'StationEncoded', 'ProduitEncoded',
    'mois', 'semaine_annee', 'est_fin_mois',
    'Lag_1', 'Lag_7',
    'Rolling_4sem', 'Rolling_12sem', 'Rolling_Std_4sem',
    'nb_commandes',
]
TARGET = 'quantite_semaine'

print(f"Lignes avant nettoyage : {len(df_weekly):,}")

df_ml = df_weekly.dropna(subset=FEATURES).reset_index(drop=True)

print(f"Lignes après nettoyage : {len(df_ml):,}")
print(f"Lignes supprimées      : {len(df_weekly) - len(df_ml):,}")
print(f"NaN restants           : {df_ml[FEATURES].isnull().sum().sum()}")
print()

# Aperçu final
df_ml[['semaine', 'Code Station', 'N° article', TARGET] + FEATURES].head(5)

Lignes avant nettoyage : 20,207
Lignes après nettoyage : 18,918
Lignes supprimées      : 1,289
NaN restants           : 0



,semaine,Code Station,N° article,quantite_semaine,StationEncoded,ProduitEncoded,mois,semaine_annee,est_fin_mois,Lag_1,Lag_7,Rolling_4sem,Rolling_12sem,Rolling_Std_4sem,nb_commandes
0,2023-01-16,ST0001,A0001,10000,0,0,1,3,0,12000.0,20000.0,17750.0,16428.571429,11026.483271,1
1,2023-01-23,ST0001,A0001,10000,0,0,1,4,0,10000.0,8000.0,16500.0,15625.000000,11704.699911,1
2,2023-01-30,ST0001,A0001,14000,0,0,1,5,1,10000.0,16000.0,10500.0,15000.000000,1000.000000,1
3,2023-02-06,ST0001,A0001,14000,0,0,2,6,0,14000.0,15000.0,11500.0,14900.000000,1914.854216,1
4,2023-02-13,ST0001,A0001,10000,0,0,2,7,0,14000.0,34000.0,12000.0,14818.181818,2309.401077,1


In [71]:
# Vérifier les trous dans la série pour une station
mask = (df_ml['Code Station'] == 'ST0001') & (df_ml['N° article'] == 'A0001')

serie = df_ml[mask][['semaine', 'quantite_semaine', 'Lag_1', 'Lag_7']].head(15)
serie['ecart_jours'] = serie['semaine'].diff().dt.days
print(serie.to_string())

      semaine  quantite_semaine    Lag_1    Lag_7  ecart_jours
0  2023-01-16             10000  12000.0  20000.0          NaN
1  2023-01-23             10000  10000.0   8000.0          7.0
2  2023-01-30             14000  10000.0  16000.0          7.0
3  2023-02-06             14000  14000.0  15000.0          7.0
4  2023-02-13             10000  14000.0  34000.0          7.0
5  2023-02-20              8000  10000.0  10000.0          7.0
6  2023-02-27             12000   8000.0  12000.0          7.0
7  2023-03-06             20000  12000.0  10000.0          7.0
8  2023-03-13             20000  20000.0  10000.0          7.0
9  2023-03-20             20000  20000.0  14000.0          7.0
10 2023-03-27             20000  20000.0  14000.0          7.0
11 2023-04-03             20000  20000.0  10000.0          7.0
12 2023-04-10             28000  20000.0   8000.0          7.0
13 2023-05-08             45000  28000.0  12000.0         28.0
14 2023-05-15             40000  45000.0  20000.0      

## 11. Sauvegarde du dataset final

In [73]:
# SAUVEGARDE ET EXPORT 
df_ml.to_csv("../data/propre/dataset_ml.csv", index=False, encoding="utf-8-sig")
print("Dataset exporté : data_ml_ready.csv")

Dataset exporté : data_ml_ready.csv


**Ce qu'on a fait et pourquoi**
On a d'abord agrégé les données en semaines. Plutôt que de travailler jour par jour (avec des trous partout car les stations ne commandent pas tous les jours), on a regroupé toutes les livraisons par semaine pour chaque station et chaque produit. Résultat : une série régulière et propre où chaque ligne représente "combien de litres la station X a commandé durant la semaine Y".  

Ensuite on a créé les features temporelles — le mois, le numéro de semaine, l'indicateur fin de mois. Ces features donnent au modèle le sens du temps : il sait qu'on est en janvier, en semaine 3, en fin de mois. Sans ça, le modèle verrait juste des chiffres sans contexte calendaire.
Puis on a créé les lags. Lag_1 dit au modèle "la semaine dernière cette station a commandé X litres". Lag_7 dit "il y a 7 semaines elle avait commandé Y litres". C'est la mémoire de la série — le modèle apprend que la demande passée prédit la demande future.
Ensuite les rolling windows. Rolling_4sem est la moyenne des 4 dernières semaines — elle capture la tendance récente. Rolling_12sem est la moyenne des 12 dernières semaines — elle capture la tendance de fond sur 3 mois. Rolling_Std_4sem mesure l'instabilité récente de la station : une valeur élevée signale une station dont la consommation est erratique et donc à risque de rupture.
Enfin on a encodé les stations et les produits en nombres pour que XGBoost et Random Forest puissent les lire, et on a supprimé les NaN générés par les lags et rolling en début de série.
Ce qu'on a au final
Un dataset propre de data_ml_ready.csv où chaque ligne contient tout ce qu'un modèle a besoin de savoir pour prédire la demande de la semaine suivante d'une station donnée — son historique récent, sa tendance, sa volatilité, et le contexte calendaire. C'est exactement ce que XGBoost et Random Forest vont consommer pour apprendre et générer des prévisions.
Le lien avec ton mémoire
Cette étape de feature engineering est ce qui transforme un simple historique de commandes en un système capable d'anticiper les besoins. C'est le cœur technique de ton système d'aide à la décision — sans ces features, le modèle n'aurait aucun moyen de distinguer une semaine normale d'une semaine de forte demande, ni de reconnaître qu'une station est en train d'accélérer sa consommation.